NetID: leahnl2

Assignment: Lab 8

Grade: _ / 9

Comments:


# CS448 - Lab 8: Audio Classification 

## Part 1: Making a speech detector

In this section we will design a simple classifier that will let us know if its input is speech or non-speech. Download the data archive from: [ https://drive.google.com/file/d/1Z8tj-HHQCvT54Mr20Dc1UfV48SPY449r/view ]. In this part we will use the dataset in data/SpeechMusic. In it you will find two directories, `speech/` and `music/` containing data from each class.

*Note: if you keep this archive in the same directory as your code, either keep the original name "`data`" or write the new folder name in`.gitignore` so you don't accidentally push the data to your repository!*

Randomly select 50 soundfiles from each directory to use as training data, and use the remaining sounds as testing data. For all of the sounds we will compute a representation that makes the classification easier and we will use a simple Gaussian model to classify them. Do the following:

- Perform an STFT for each sound, take it’s magnitude and raise it to 0.3 to improve contrast
    - We will consider each spectral slice of that to be a data point
- Using the training data of each sound:
    - Calculate the mean column and the diagonal covariance of the columns
    - You will thus get two sets of Gaussian parameters that model each sound class
- For each testing data point:
    - Calculate the likelihood of each column based on the above models
	- To calculate the entire file likelihood add all the frame likelihoods
	- Assign each soundfile to the class that gets the highest likelihood

For extra credit implement the parameter estimation and model likelihood yourself. If you are too lazy for that you can instead use ```sklearn.mixture.GaussianMixture``` to learn a diagonal single-Gaussian model per class.

How do the results look like? If you rerun this with a different training/testing set, is there an appreciable difference? On average over multiple training/testing sets what accuracy do you get?

In [1]:
import utils
import numpy as np
import matplotlib.pyplot as plt
import scipy
import librosa

In [2]:
# generate training_count random samples out of the total_count
def split_data(total_count, training_count):
    selections = np.zeros(total_count)
    
    # randomly select soundfiles from given directory for training
    count = 0
    while count < training_count:
        random_num = np.random.randint(1, total_count + 1) # [1, total_count]
    
        if selections[random_num - 1] == 0:
            selections[random_num - 1] = 1
            count += 1

    return selections

# return the training and testing data based on selections
def separate_data(selections, path):

    # define STFT params
    dft_size = 512
    hop_size = dft_size // 4
    hann_window = np.hanning(dft_size)
    freq_bins = (dft_size // 2) + 1

    # intitialize training and testing data
    training_list = []
    testing_list = []
    
    for i in range(len(selections)):
        # load in the sound
        sound_path = path + str(i+1) + ".wav"
        sound_sr, sound_x = utils.wavreadfile(sound_path)
    
        # compute STFT of sound and improve contrast
        stft = utils.stft(sound_x, dft_size, hop_size, hann_window, 0)
        stft = np.abs(stft) ** 0.3
            
        # separate into training or testing data
        if selections[i] == 1:
            training_list.append(stft.T)
        else:
            testing_list.append(stft.T)

    training_data = np.vstack(training_list)
    testing_data = np.array(testing_list)

    return training_data, testing_data

# learn the mean and covariance Gaussian parameters
def learn_gaussian_parameters(training_data):
    mean = np.mean(training_data, axis=0)
    covariance = np.var(training_data, axis=0) + 1e-3 # add small value to prevent divide by 0
    
    return (mean, covariance)

# assign testing data slices to class based on calculated likelihood
def calculate_likelihood(params, x):
    mean, cov = params
    
    # use log likelihood of equation P(x|m, C) = (1 / sqrt((2pi)^k * |C|) e ** (-0.5 (x - m).T * C^(-1) * (x - m))
    
    cov_inv = 1 / cov # C^(-1) 
    k = len(mean) # dimensions = number of frequency bins
    sum_log_cov = np.sum(np.log(cov)) # sum(i=1, k) (ln(covariance))
    
    diff_squared = (x - mean) ** 2        
    log_likelihood = -0.5 * (k * np.log(2 * np.pi) + sum_log_cov + np.sum(diff_squared * cov_inv))
        
    return log_likelihood

# calculate the likelihoods of each data point of each sound and return results
def classify_data(music_params, speech_params, testing_data):

    num_samples, num_frames, _ = np.shape(testing_data)
    classifications = []

    # classify each sound
    for i in range(num_samples): 

        music_likelihood = 0
        speech_likelihood = 0
        
        # likelihoods = np.zeros(num_frames) # 0 for music, 1 for speech
        for j in range(num_frames):

            # get spectral slice (data point)
            x = testing_data[i, j, :]

            # calculate likelihoods for both classes
            music_likelihood += calculate_likelihood(music_params, x)
            speech_likelihood += calculate_likelihood(speech_params, x)

        # determine the sound's classification
        if music_likelihood > speech_likelihood:
            classifications.append("Music")
        else:
            classifications.append("Speech")

    return classifications
    

In [3]:
# get file paths
music_path = "./data/SpeechMusic/music/"
speech_path = "./data/SpeechMusic/speech/"

In [64]:
# YOUR CODE HERE
accuracy = 0
trials = 15

for i in range(trials):

    # make random selections of training and testing data samples
    music_selections = split_data(60, 50)
    speech_selections = split_data(60, 50)
    
    # separate into training and testing data
    music_training, music_testing = separate_data(music_selections, music_path)
    speech_training, speech_testing = separate_data(speech_selections, speech_path)

    # calculate the Gaussian parameters for both training sets
    music_params = learn_gaussian_parameters(music_training)
    speech_params = learn_gaussian_parameters(speech_training)

    # classify the testing data for each type of sound 
    music_classifications = classify_data(music_params, speech_params, music_testing)
    speech_classifications = classify_data(music_params, speech_params, speech_testing)

    # calculate accuracy
    num_testing_data = len(music_testing)
    curr_accuracy = ((music_classifications.count("Music") + speech_classifications.count("Speech")) / (2 * num_testing_data))

    # make nice print-out for trials (only the first few)
    if i < 3:
        print("###########")
        print(f"# Trial {i+1} #")
        print("###########")
        print("Classifications for music test data:", music_classifications)
        print("Classifications for speech test data:", speech_classifications)
        print("Accuracy: ", curr_accuracy)
        print("\n")

    accuracy += curr_accuracy

if trials > 3:
    print("...\n")

# calculate average accuracy
accuracy /= trials
print(f"Average overall accuracy for {trials} trials: ", accuracy)

###########
# Trial 1 #
###########
Classifications for music test data: ['Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Speech', 'Speech', 'Speech']
Classifications for speech test data: ['Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech']
Accuracy:  0.85


###########
# Trial 2 #
###########
Classifications for music test data: ['Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Speech', 'Music', 'Speech']
Classifications for speech test data: ['Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech']
Accuracy:  0.9


###########
# Trial 3 #
###########
Classifications for music test data: ['Music', 'Music', 'Music', 'Music', 'Speech', 'Music', 'Music', 'Music', 'Music', 'Music']
Classifications for speech test data: ['Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Speech', 'Music']
Accuracy:  0.9


...

Average overall accuracy for 

Running multiple trials with different randomized training/testing datasets resulted in classifications accuracies usually between 0.7 to 0.9. The average accuracy after running 15 trials was 0.7999999999999999. There weren't drastic differences between different training/testing datasets, although the accuracies could vary up to 20%. Some of the random samples seem easier to classify than others based on this observation.

## Part 2: Making a music genre classifier

We will repeat the above, but this time we will perform music genre classification. To do so we will use a slightly more elaborate feature representation, and a stronger classification model. If you downloaded the data archive pointed to above, you will find a subset of the CTZAN dataset in the data/genre folder, this is a benchmark data set for music genre classification.

Just as before, you will find a set of directories with examples of each sound class that we want to recognize. For each class, split the soundfiles into a training set (50% of data) and testing set (remaining 50% of data).

For a representation we will use MFCC features. For extra credit, code these yourself otherwise you can use the implementation from the ```librosa``` library. Once all the files are transformed we will have a series of MFCC frames for each recording (as opposed to spectral frames as is in the case of the STFT). We will use these as the data to classify.

For each class learn a Gaussian model (with a diagonal covariance again). This will be the same process as above.
In order to evaluate how good this works we will use the following procedure. For each sound in the training data, get the likelihood of each MFCC frame based on the learned Gaussian models and sum these over the entire file just as we did before. Use the resulting values to get a classification result for each . Report how accurate your results are. Now report the accuracy using your testing data instead.

Now will use a better classifier to hopefully get better accuracy. We will use a Gaussian Mixture Model (```sklearn.mixture.GaussianMixture```). Just as before you should learn one such model for each class using the corresponding training data.

How many Gaussians do you need in your GMM to get the best results? Do the MFCC parameters make a difference? Play around with the numbers to get the best possible results.  You should be able to get at least 70% accuracy on average.

In [3]:
# return the training and testing data as MFCC features
def compute_mfcc_features(path, music_type, num_mfcc=40):

    # separate the data into training and testing sets (50 each)
    selections = split_data(100, 50)

    # intitialize training and testing data
    training_list = []
    testing_list = []
    
    for i in range(len(selections)):
        # load in the sound
        sound_path = path + music_type + "." + str(i).zfill(5) + ".mp3"
        sound_sr, sound_x = utils.mp3readfile(sound_path)
    
        # compute MFCC of sound using librosa library
        mfcc = librosa.feature.mfcc(y=sound_x, sr=sound_sr, n_mfcc=num_mfcc)
        mfcc = mfcc.T # (frames, mfcc index)
        # print(mfcc.shape)

        # separate into training or testing data
        if selections[i] == 1:
            training_list.append(mfcc)
        else:
            testing_list.append(mfcc)

    return training_list, testing_list
    
# calculate the likelihoods of each data point of each sound and return results
def classify_genre(genre_params, genre_names, data):

    num_samples = len(data)
    classifications = []

    # classify each sound
    for i in range(num_samples): 

        mfcc_matrix = data[i]
        genre_likelihoods = {}

        # calculate summed likelihood for each genre
        for genre, params in zip(genre_names, genre_params):
            total_likelihood = 0

            # get each MFCC frame
            for x in mfcc_matrix:
                total_likelihood += calculate_likelihood(params, x)

            genre_likelihoods[genre] = total_likelihood
        
        # get the classified genre (largest likelihood sum)
        classified_genre = max(genre_likelihoods, key=genre_likelihoods.get)
        classifications.append(classified_genre)

    return classifications

In [4]:
# get file paths
classical_path = "./data/genres/classical/"
disco_path = "./data/genres/disco/"
metal_path = "./data/genres/metal/"
pop_path = "./data/genres/pop/"
reggae_path = "./data/genres/reggae/"

### Simple Gaussian Model Code

In [12]:
# YOUR CODE HERE

genre_names =  ["classical", "disco", "metal", "pop", "reggae"]
overall_training_accuracies = np.zeros(5, dtype=np.float32)
overall_testing_accuracies = np.zeros(5, dtype=np.float32)

# perform multiple trials to get average accuracies
num_trials = 10
for i in range(num_trials):

    # make random selections of training and testing data samples
    classical_training, classical_testing = compute_mfcc_features(classical_path, "classical")
    disco_training, disco_testing = compute_mfcc_features(disco_path, "disco")
    metal_training, metal_testing = compute_mfcc_features(metal_path, "metal")
    pop_training, pop_testing = compute_mfcc_features(pop_path, "pop")
    reggae_training, reggae_testing = compute_mfcc_features(reggae_path, "reggae")
    
    # calculate the Gaussian parameters for all training sets
    classical_params = learn_gaussian_parameters(np.vstack(classical_training))
    disco_params = learn_gaussian_parameters(np.vstack(disco_training))
    metal_params = learn_gaussian_parameters(np.vstack(metal_training))
    pop_params = learn_gaussian_parameters(np.vstack(pop_training))
    reggae_params = learn_gaussian_parameters(np.vstack(reggae_training))

    genre_params = (classical_params, disco_params, metal_params, pop_params, reggae_params)

    # classify the training data for each type of sound 
    training_data_set = [classical_training, disco_training, metal_training, pop_training, reggae_training]
    training_accuracies = np.zeros(5, dtype=np.float32)

    idx = 0
    for genre, training_data in zip(genre_names, training_data_set):
        classifications = classify_genre(genre_params, genre_names, training_data)
        curr_accuracy = classifications.count(genre) / len(training_data)
        training_accuracies[idx] = curr_accuracy
        idx += 1
    
    overall_training_accuracies += training_accuracies

    # classify the testing data for each type of sound 
    testing_data_set = [classical_testing, disco_testing, metal_testing, pop_testing, reggae_testing]
    testing_accuracies = np.zeros(5, dtype=np.float32)

    idx = 0
    for genre, testing_data in zip(genre_names, testing_data_set):
        classifications = classify_genre(genre_params, genre_names, testing_data)
        curr_accuracy = classifications.count(genre) / len(testing_data)
        testing_accuracies[idx] = curr_accuracy
        idx += 1
    
    overall_testing_accuracies += testing_accuracies

# compute overall accuracies
overall_training_accuracies /= num_trials
overall_testing_accuracies /= num_trials

overall_training_accuracy = np.mean(overall_training_accuracies)
overall_testing_accuracy = np.mean(overall_testing_accuracies)


In [13]:
print("/////////////////////////////////")
print("/ Single Gaussian Model Results /")
print("/////////////////////////////////\n")

# make nice print-out for training data
training_label = f"Classification Accuracies for Training Data ({num_trials} Trials)"
print("-" * len(training_label))
print(training_label)
print("-" * len(training_label))

for genre, accuracy in zip(genre_names, overall_training_accuracies):
    print("\t" + genre.capitalize() + f" accuracy: {accuracy}")

print(f"\nOverall accuracy: {overall_training_accuracy}\n")

# make nice print-out for testing data
testing_label = f"Classification Accuracies for Testing Data ({num_trials} Trials)"
print("-" * len(testing_label))
print(testing_label)
print("-" * len(testing_label))

for genre, accuracy in zip(genre_names, overall_testing_accuracies):
    print("\t" + genre.capitalize() + f" accuracy: {accuracy}")

print(f"\nOverall accuracy: {overall_testing_accuracy}")

/////////////////////////////////
/ Single Gaussian Model Results /
/////////////////////////////////

-------------------------------------------------------
Classification Accuracies for Training Data (10 Trials)
-------------------------------------------------------
	Classical accuracy: 0.9259999990463257
	Disco accuracy: 0.5479999780654907
	Metal accuracy: 0.9160000085830688
	Pop accuracy: 0.8640000224113464
	Reggae accuracy: 0.7639999389648438

Overall accuracy: 0.8035999536514282

------------------------------------------------------
Classification Accuracies for Testing Data (10 Trials)
------------------------------------------------------
	Classical accuracy: 0.9199999570846558
	Disco accuracy: 0.5180000066757202
	Metal accuracy: 0.8880000114440918
	Pop accuracy: 0.8579999804496765
	Reggae accuracy: 0.7099999785423279

Overall accuracy: 0.7788000106811523


### Gaussian Mixture Model Code

In [5]:
import sklearn

# Learn the GMMs for each genre's training data
def train_gmm_models(training_data_set, components):

    genre_models = []
    for training_data in training_data_set:
        # stack features from all sounds together
        training_data = np.vstack(training_data) 

        # learn the Gaussian Mixture Model and fit it to the genre's training data
        gmm = sklearn.mixture.GaussianMixture(n_components=components, covariance_type='diag', random_state=11).fit(training_data)  
        genre_models.append(gmm)
        
    return genre_models

# calculate the likelihoods of each data point of each sound using GMM and return results
def classify_genre_with_gmm(genre_models, genre_names, data):
    
    num_samples = len(data)
    classifications = []

    # classify each sound
    for i in range(num_samples):
        
        mfcc_matrix = data[i]
        genre_likelihoods = {}
    
        # calculate summed likelihood for each genre
        for genre, model in zip(genre_names, genre_models):
            log_likelihood = model.score_samples(mfcc_matrix)
            genre_likelihoods[genre] = np.sum(log_likelihood)
        
        # get the classified genre (largest likelihood sum)
        classified_genre = max(genre_likelihoods, key=genre_likelihoods.get)
        classifications.append(classified_genre)

    return classifications


In [10]:
genre_names =  ["classical", "disco", "metal", "pop", "reggae"]
overall_training_accuracies = np.zeros(5, dtype=np.float32)
overall_testing_accuracies = np.zeros(5, dtype=np.float32)

# perform multiple trials to get average accuracies
num_trials = 10
num_gaussian_components = 5

for i in range(num_trials):

    # make random selections of training and testing data samples
    classical_training, classical_testing = compute_mfcc_features(classical_path, "classical")
    disco_training, disco_testing = compute_mfcc_features(disco_path, "disco")
    metal_training, metal_testing = compute_mfcc_features(metal_path, "metal")
    pop_training, pop_testing = compute_mfcc_features(pop_path, "pop")
    reggae_training, reggae_testing = compute_mfcc_features(reggae_path, "reggae")
    
    # train the Gaussian models for all training sets
    training_data_set = [classical_training, disco_training, metal_training, pop_training, reggae_training]
    genre_models = train_gmm_models(training_data_set, num_gaussian_components)

    # classify the training data for each type of sound 
    training_accuracies = np.zeros(5, dtype=np.float32)

    idx = 0
    for genre, training_data in zip(genre_names, training_data_set):
        classifications = classify_genre_with_gmm(genre_models, genre_names, training_data)
        curr_accuracy = classifications.count(genre) / len(training_data)
        training_accuracies[idx] = curr_accuracy
        idx += 1
    
    overall_training_accuracies += training_accuracies

    # classify the testing data for each type of sound 
    testing_data_set = [classical_testing, disco_testing, metal_testing, pop_testing, reggae_testing]
    testing_accuracies = np.zeros(5, dtype=np.float32)

    idx = 0
    for genre, testing_data in zip(genre_names, testing_data_set):
        classifications = classify_genre_with_gmm(genre_models, genre_names, testing_data)
        curr_accuracy = classifications.count(genre) / len(testing_data)
        testing_accuracies[idx] = curr_accuracy
        idx += 1
    
    overall_testing_accuracies += testing_accuracies

# compute overall accuracies
overall_training_accuracies /= num_trials
overall_testing_accuracies /= num_trials

overall_training_accuracy = np.mean(overall_training_accuracies)
overall_testing_accuracy = np.mean(overall_testing_accuracies)


In [11]:
print("//////////////////////////////////")
print("/ Gaussian Mixture Model Results /")
print("//////////////////////////////////\n")

# make nice print-out for training data
training_label = f"Classification Accuracies for Training Data ({num_trials} Trials)"
print("-" * len(training_label))
print(training_label)
print("-" * len(training_label))

for genre, accuracy in zip(genre_names, overall_training_accuracies):
    print("\t" + genre.capitalize() + f" accuracy: {accuracy}")

print(f"\nOverall accuracy: {overall_training_accuracy}\n")

# make nice print-out for testing data
testing_label = f"Classification Accuracies for Testing Data ({num_trials} Trials)"
print("-" * len(testing_label))
print(testing_label)
print("-" * len(testing_label))

for genre, accuracy in zip(genre_names, overall_testing_accuracies):
    print("\t" + genre.capitalize() + f" accuracy: {accuracy}")

print(f"\nOverall accuracy: {overall_testing_accuracy}")

//////////////////////////////////
/ Gaussian Mixture Model Results /
//////////////////////////////////

-------------------------------------------------------
Classification Accuracies for Training Data (10 Trials)
-------------------------------------------------------
	Classical accuracy: 0.9899999499320984
	Disco accuracy: 0.7879999876022339
	Metal accuracy: 0.956000030040741
	Pop accuracy: 0.9479999542236328
	Reggae accuracy: 0.8079999685287476

Overall accuracy: 0.8979999423027039

------------------------------------------------------
Classification Accuracies for Testing Data (10 Trials)
------------------------------------------------------
	Classical accuracy: 0.9519999623298645
	Disco accuracy: 0.6439999341964722
	Metal accuracy: 0.9020000696182251
	Pop accuracy: 0.8960000276565552
	Reggae accuracy: 0.7640000581741333

Overall accuracy: 0.83160001039505


I found that using 4-5 Gaussian components resulted in the best results. Higher values resulted in Gaussian models that were too complex for the data, while smaller values resulted in Gaussian models that were too simple. The MFCC parameter n_mfcc controlled the number of MFCCs produced for a given sound. I tried to play around with values that would make my feature slices better for classification (finding the balance between the data under/over-respresenting the sound's features). I found that using larger values (between 20 to 40), resulted in higher accuracies.

## Part 3: Make it better (extra credit, required for 4-hour credit)

There is no shortage of techniques (and free code) to use for classification. Revisit the two problems above and use any other type of classifier you want (Neural Nets, Boosting, Decision Trees, whatever). Also feel free to use any feature you want. Can you improve on the results you got before? How much higher can you get your accuracy for either case?

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()